#### MAI202 - Assignment2 -# Weather Forecasting with Sequence Models
##### Monireh Eshghinezhad

In [ ]:
import importlib.util
if importlib.util.find_spec("torchmetrics") is None:
    %pip install -q torchmetrics

import sys, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torchmetrics
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=12)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device: use a GPU/accelerator if available 
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")


Using device: cpu


### 1. Load & Explore the Dataset

In [3]:
df = pd.read_csv("weather_prediction_dataset.csv")
df["DATE"] = pd.to_datetime(df["DATE"], format="%Y%m%d")
df = df.set_index("DATE").sort_index()
dgf= df.drop_duplicates
print(f"Dataset shape: {df.shape}")
print(f"{len(df)} daily records from {df.index.min().date()} to {df.index.max().date()}")
print(f"Missing values: {df.isna().sum().sum()}")


Dataset shape: (3654, 164)
3654 daily records from 2000-01-01 to 2010-01-01
Missing values: 0


 #### Scenario 1: Single-step (14 → 1) 
 Feature: BASEL_temp_mean               
 Slide a 14-day window, predict the next 1 day.